# Day 7 — Environmental Effects and Temperature Features

## Goal

Explore whether sea temperature is associated with lice levels and future breach risk, and create temperature-based warning features.

In [44]:
# Load libraries
import pandas as pd
from pathlib import Path

In [117]:
feature_file = (
    Path.home()
    / "Documents"
    / "Kazi_Academic"
    / "Projects"
    / "Aquaculture"
    / "fish-health-analytics"
    / "data"
    / "processed"
    / "barentswatch_lice_2025_features.csv"
)

df = pd.read_csv(feature_file)

/tmp/ipykernel_116564/125858662.py:13: DtypeWarning: Columns (0: weekly_lice_limit) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(feature_file)


In [118]:
# Check basic information about sea temperature
df["sea_temperature_c"].describe()

count    31027.000000
mean         9.670771
std          3.947356
min          0.300000
25%          6.600000
50%          9.350000
75%         12.660000
max        196.180000
Name: sea_temperature_c, dtype: float64

In [119]:
# Count missing temperature values
df["sea_temperature_c"].isna().sum()

np.int64(24691)

In [120]:
# Percentage of missing temperature values
missing_temp_percent = (
    df["sea_temperature_c"].isna().mean() * 100
)

missing_temp_percent

np.float64(44.31422520549912)

## Next: inspect the suspicious values

* First, show the 10 highest temperatures:

In [121]:
# Sort temperature from highest to lowest
highest_temp = df.sort_values(
    "sea_temperature_c",
    ascending=False
)

# Show the highest 10 values
highest_temp[
    [
        "locality_name",
        "year",
        "week",
        "sea_temperature_c"
    ]
].head(10)

,locality_name,year,week,sea_temperature_c
19828,Djupedalen,2025,24,196.18
21014,Oterstegdalen,2025,42,142.00
7229,Storøya Nø,2025,9,98.00
10667,Bukkeholmen,2025,15,80.10
26543,Tårnvika,2025,7,61.24
43856,Forrahammaren,2025,41,56.20
28078,Stokkvika,2025,34,46.20
44756,Gisløy Nø,2025,5,43.70
15027,Isane,2025,7,40.90
52784,Lingaholmane,2025,26,36.33


In [122]:
# Then count how many temperatures are above 25°C:
# Find unusually high sea temperatures
high_temp = df[
    df["sea_temperature_c"] > 25
]

# Count them
high_temp.shape[0]

27

In [123]:
high_temp[
    [
        "locality_id",
        "locality_name",
        "year",
        "week",
        "sea_temperature_c"
    ]
].sort_values(
    "sea_temperature_c",
    ascending=False
)

,locality_id,locality_name,year,week,sea_temperature_c
19828,13229,Djupedalen,2025,24,196.18
21014,13345,Oterstegdalen,2025,42,142.00
7229,11298,Storøya Nø,2025,9,98.00
10667,11721,Bukkeholmen,2025,15,80.10
26543,16165,Tårnvika,2025,7,61.24
43856,33697,Forrahammaren,2025,41,56.20
28078,19015,Stokkvika,2025,34,46.20
44756,34357,Gisløy Nø,2025,5,43.70
15027,12224,Isane,2025,7,40.90
52784,45066,Lingaholmane,2025,26,36.33


In [124]:
# Create a quality flag.
# Start by assuming temperature is OK
df["temperature_flag"] = "OK"

# Clearly unrealistic temperature
df.loc[
    df["sea_temperature_c"] > 30,
    "temperature_flag"
] = "Invalid"

# High temperature that needs checking
df.loc[
    (df["sea_temperature_c"] > 25) &
    (df["sea_temperature_c"] <= 30),
    "temperature_flag"
] = "Check"

In [125]:
df["temperature_flag"].value_counts()

temperature_flag
OK         55691
Invalid       15
Check         12
Name: count, dtype: int64

## inspect the suspicious 25–30°C values

In [126]:
# Keep only temperatures that need checking
temp_check = df[
    df["temperature_flag"] == "Check"
]

# Show them
temp_check[
    [
        "locality_id",
        "locality_name",
        "week",
        "sea_temperature_c"
    ]
].sort_values(
    "sea_temperature_c",
    ascending=False
)

,locality_id,locality_name,week,sea_temperature_c
28070,19015,Stokkvika,26,27.31
11876,11813,Svinøy V,28,26.43
48787,37177,Djuptaren,33,26.41
11470,11783,Hella,38,26.37
7257,11298,Storøya Nø,37,26.11
18649,13020,Nygård,41,26.03
29905,20776,Storoksen,41,25.61
10264,11667,Gardskråneset,28,25.39
25120,14435,Ljøsøy N,40,25.39
34226,26935,Frovågneset,31,25.29


### Are these values isolated spikes, or do the neighboring weeks also show unusually warm water?

* Start with the highest one, Stokkvika, week 26 = 27.31°C.

In [127]:
# Select Stokkvika
stokkvika = df[df["locality_id"] == 19015]

# Keep only week and temperature
stokkvika_temp = stokkvika[
    ["week", "sea_temperature_c"]
]

# Sort by week
stokkvika_temp = stokkvika_temp.sort_values("week")

# Show weeks around week 26
stokkvika_temp[
    (stokkvika_temp["week"] >= 23) &
    (stokkvika_temp["week"] <= 29)
]

,week,sea_temperature_c
28067,23,NaN
28068,24,NaN
28069,25,9.40
28070,26,27.31
28071,27,12.03
28072,28,13.15
28073,29,34.20


In [128]:
# Rather than manually checking every locality one by one, let's create a clean temperature column while preserving the original data.
# Make a copy of the original temperature
df["sea_temperature_clean"] = df["sea_temperature_c"]

In [129]:
# Remove clearly impossible values above 30°C:
# Replace clearly invalid temperatures with NaN
df.loc[
    df["sea_temperature_clean"] > 30,
    "sea_temperature_clean"
] = pd.NA

In [130]:
# Create a column to flag suspicious high temperatures
df["high_temperature_check"] = False

df.loc[
    (df["sea_temperature_clean"] > 25) &
    (df["sea_temperature_clean"] <= 30),
    "high_temperature_check"
] = True

In [131]:
df.loc[
    (df["locality_id"] == 19015) &
    (df["week"] == 26),
    "sea_temperature_clean"
] = pd.NA

# check the remaining 25–30°C values automatically using their neighboring weeks

In [132]:
# Sort each locality by year and week
df = df.sort_values(
    ["locality_id", "year", "week"]
).copy()

In [133]:
# Group temperature by locality
temp_by_locality = df.groupby("locality_id")["sea_temperature_clean"]

# Previous week's temperature
df["temp_previous_week"] = temp_by_locality.shift(1)

In [134]:
# Group temperature by locality
temp_by_locality = df.groupby("locality_id")["sea_temperature_clean"]

# Previous week's temperature
df["temp_previous_week"] = temp_by_locality.shift(1)

In [135]:
# Next week's temperature
df["temp_next_week"] = temp_by_locality.shift(-1)

In [136]:
# Select temperatures that need checking
temp_check = df[
    (df["sea_temperature_clean"] > 25) &
    (df["sea_temperature_clean"] <= 30)
]

In [137]:
columns_to_show = [
    "locality_name",
    "week",
    "temp_previous_week",
    "sea_temperature_clean",
    "temp_next_week"
]

temp_check[columns_to_show]

,locality_name,week,temp_previous_week,sea_temperature_clean,temp_next_week
7257,Storøya Nø,37,12.94,26.11,13.36
10164,Gråvika,32,15.14,25.14,14.24
10264,Gardskråneset,28,12.63,25.39,12.89
11470,Hella,38,15.01,26.37,15.50
11876,Svinøy V,28,14.04,26.43,15.37
18649,Nygård,41,14.97,26.03,14.87
25120,Ljøsøy N,40,14.53,25.39,13.60
29905,Storoksen,41,13.81,25.61,12.53
31866,Nordbotnet,40,12.37,25.20,11.81
34226,Frovågneset,31,13.26,25.29,16.16


# Make the cleaning reproducible
* Instead of manually deleting each locality, create a rule based on the neighboring weeks.

In [138]:
# Step 1: Calculate the average of previous and next week
df["temp_neighbor_mean"] = (
    df["temp_previous_week"] + df["temp_next_week"]
) / 2

In [139]:
# Step 2: Difference from surrounding weeks
df["temp_difference"] = (
    df["sea_temperature_clean"] - df["temp_neighbor_mean"]
)

In [140]:
# Step 3: Identify suspicious temperature spikes
suspicious_spike = (
    (df["sea_temperature_clean"] > 25) &
    (df["temp_difference"] > 5)
)

In [141]:
# Step 4: Replace suspicious spikes with missing values
df.loc[
    suspicious_spike,
    "sea_temperature_clean"
] = pd.NA

In [142]:
suspicious_spike.sum()

np.int64(11)

In [143]:
df["sea_temperature_clean"].describe()

count    31000.000000
mean         9.638128
std          3.608306
min          0.300000
25%          6.600000
50%          9.340000
75%         12.640000
max         24.750000
Name: sea_temperature_clean, dtype: float64

### Is sea temperature associated with adult female lice levels?
* Keep only rows where both temperature and lice were measured.

In [144]:
# keep rows wit valid temperature and lice valiues
temp_lice_data = df[df["sea_temperature_clean"].notna()& df["adult_female_lice"].notna()].copy()

In [145]:
#check how many observations remanin
temp_lice_data.shape

(30272, 38)

In [146]:
# Correlation
correlation = temp_lice_data[["sea_temperature_clean", "adult_female_lice"]].corr()
correlation

,sea_temperature_clean,adult_female_lice
sea_temperature_clean,1.00000,0.14785
adult_female_lice,0.14785,1.00000


In [147]:
# divide temperature into ranges
# Step 1: Create temperature groups
temp_lice_data["temperature_group"] = pd.cut(
    temp_lice_data["sea_temperature_clean"],
    bins=[0, 5, 10, 15, 20, 25],
    labels=[
        "0-5°C",
        "5-10°C",
        "10-15°C",
        "15-20°C",
        "20-25°C"
    ]
)

In [148]:
# calculate the average lice level in each temperature group
# Step 2: Calculate average lice for each temperature group
temperature_summary = (
    temp_lice_data
    .groupby("temperature_group", observed=True)
    ["adult_female_lice"]
    .mean()
)

temperature_summary

temperature_group
0-5°C      0.145605
5-10°C     0.141534
10-15°C    0.215471
15-20°C    0.254853
20-25°C    0.156809
Name: adult_female_lice, dtype: float64

In [149]:
# count how many observations each group contains.
# Step 3: Count observations in each temperature group
temperature_counts = (
    temp_lice_data["temperature_group"]
    .value_counts()
    .sort_index()
)

temperature_counts


temperature_group
0-5°C       2710
5-10°C     14085
10-15°C    11419
15-20°C     2011
20-25°C       47
Name: count, dtype: int64

* In this dataset, adult female lice levels tend to be higher in moderately warm water, particularly around 10–20°C, while the >20°C range is too sparsely sampled to interpret confidently.

# Is sea temperature different before a future breach compared with a future non-breach?


In [150]:

# Keep rows where future breach status is known
future_temp_data = df[
    df["future_breach"].notna() &
    df["sea_temperature_clean"].notna()
].copy()

In [151]:
future_temp_data.groupby(
    "future_breach"
)["sea_temperature_clean"].mean()

future_breach
0.0     9.630187
1.0    11.978348
Name: sea_temperature_clean, dtype: float64

In [152]:
future_temp_data["future_breach"].value_counts()

future_breach
0.0    28127
1.0     1041
Name: count, dtype: int64

# feature: previous-week temperature

In [153]:
# Step1: Sort each locality by time.
df = df.sort_values(["locality_id", "year", "week"]).copy()

In [154]:
# Group temperature by locality
temperature_by_locality = df.groupby("locality_id") ["sea_temperature_clean"]
temperature_by_locality

In [155]:
# Take the previous week temperature
previous_temperature = temperature_by_locality.shift(1)


In [156]:
# Save it as a new features
df["temperature_lag_1"] = previous_temperature
df["temperature_lag_1"]

0        NaN
1        NaN
2        8.4
3        7.5
4        7.0
        ... 
55713    NaN
55714    NaN
55715    NaN
55716    NaN
55717    NaN
Name: temperature_lag_1, Length: 55718, dtype: float64

In [157]:
# Now calculate current temperature - previous temperature 
# Step 1: Current temperature
current_temp = df["sea_temperature_clean"]

# Step 2: Previous week's temperature
previous_temp = df["temperature_lag_1"]

# Step 3: Calculate temperature change
temp_change = current_temp - previous_temp

# Step 4: Save it as a new column
df["temperature_change"] = temp_change

In [158]:
# Inspect few row properly
columns_to_show = ["locality_name", "week", "sea_temperature_clean", "temperature_lag_1", "temperature_change"]

df[columns_to_show].head(20)

,locality_name,week,sea_temperature_clean,temperature_lag_1,temperature_change
0,NaN,52,8.6,NaN,NaN
1,Tuholmane Ø,1,8.4,NaN,NaN
2,Tuholmane Ø,2,7.5,8.4,-0.9
3,Tuholmane Ø,3,7.0,7.5,-0.5
4,Tuholmane Ø,4,6.5,7.0,-0.5
5,Tuholmane Ø,5,6.8,6.5,0.3
6,Tuholmane Ø,6,6.6,6.8,-0.2
7,Tuholmane Ø,7,6.1,6.6,-0.5
8,Tuholmane Ø,8,5.2,6.1,-0.9
9,Tuholmane Ø,9,5.8,5.2,0.6


# 3-week rolling temperature average

* This tells us the average sea temperature over the current week plus the previous two weeks.

In [159]:
# Group temperature by locality
temperature_by_locality = df.groupby("locality_id")["sea_temperature_clean"]

# calculate 3-week average temperature
temperature_mean_3w = temperature_by_locality.transform(
    lambda x:x.rolling(
        window=3, 
        min_periods=3
    ).mean()
)

# Save it as a new column
df["temperature_mean_3w"] = temperature_mean_3w

In [160]:
columns_to_show = ["locality_name", "week", "sea_temperature_clean", "temperature_lag_1", "temperature_change", "temperature_mean_3w"]

df[columns_to_show].head(20)

,locality_name,week,sea_temperature_clean,temperature_lag_1,temperature_change,temperature_mean_3w
0,NaN,52,8.6,NaN,NaN,NaN
1,Tuholmane Ø,1,8.4,NaN,NaN,NaN
2,Tuholmane Ø,2,7.5,8.4,-0.9,NaN
3,Tuholmane Ø,3,7.0,7.5,-0.5,7.633333
4,Tuholmane Ø,4,6.5,7.0,-0.5,7.000000
5,Tuholmane Ø,5,6.8,6.5,0.3,6.766667
6,Tuholmane Ø,6,6.6,6.8,-0.2,6.633333
7,Tuholmane Ø,7,6.1,6.6,-0.5,6.500000
8,Tuholmane Ø,8,5.2,6.1,-0.9,5.966667
9,Tuholmane Ø,9,5.8,5.2,0.6,5.700000


In [161]:
# Are these temperature features different before future breaches?
# Choose  the temperature features we want to compare
# Step 1: Choose the temperature features we want to compare
temperature_features = [
    "sea_temperature_clean",
    "temperature_lag_1",
    "temperature_change",
    "temperature_mean_3w"
]
# Step 2: Keep rows where future breach status is known
valid_future_data = df[
    df["future_breach"].notna()
].copy()



In [162]:
# Save the updated feature dataset
df.to_csv(
    feature_file,
    index=False
)

In [114]:
# Step 3: Compare average temperature features
temperature_comparison = (
    valid_future_data
    .groupby("future_breach")[temperature_features]
    .mean()
    .round(3)
)

temperature_comparison

,sea_temperature_clean,temperature_lag_1,temperature_change,temperature_mean_3w
future_breach,,,,
0.0,9.630,9.651,0.037,9.715
1.0,11.978,12.110,-0.094,12.126


## Summary of Day7
* Localities that breached the lice limit in the following week had higher current, previous-week, and 3-week mean sea temperatures than localities that did not breach. 
* Short-term weekly temperature change showed little evidence of being a strong warning signal.